# NH3 / H2 Qubit Hamiltonian (Colab Minimal)

This notebook now defers all logic to the repository code. Workflow:

1. Install pinned dependencies (Qiskit 2.x, qiskit-nature, PySCF).
2. Clone the repo.
3. Run the CLI script for NH3 (active space, 6 qubits) or H2 (4 qubits).
4. (Optional) Use fallback only (`--force-precomputed`) if PySCF fails.

See repository README for details and provenance notes.


In [ ]:
# === Master Environment + Repo Setup (Run FIRST) ===
import sys, subprocess, importlib, os, pathlib, numpy as np

# 1. Pin core quantum chemistry stack (idempotent)
PKGS = ['qiskit==2.1.2','qiskit-nature==0.7.2','pyscf==2.6.1','qiskit-aer']
subprocess.check_call([sys.executable,'-m','pip','install','--upgrade','--no-cache-dir']+PKGS)

# 2. Clone / update repo containing vqeskeletal.py (GroundStateFinder)
REPO_URL = 'https://github.com/Kukyos/GroundStateFinder.git'
REPO_DIR = pathlib.Path('GroundStateFinder')
if not REPO_DIR.exists():
    subprocess.check_call(['git','clone','--depth','1',REPO_URL])
else:
    try:
        subprocess.check_call(['git','-C',str(REPO_DIR),'pull','--ff-only'])
    except Exception as e:
        print('Git pull failed (continuing):', e)

# 3. Ensure repo root and src on sys.path
paths_added = []
for p in [REPO_DIR, REPO_DIR/'src']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        paths_added.append(str(p))
print('Added to sys.path:', paths_added)

# 4. Verify presence of vqeskeletal.py
vqefile = REPO_DIR/'vqeskeletal.py'
if not vqefile.exists():
    print('FATAL: vqeskeletal.py not found in cloned repo; aborting.')
else:
    print('Found vqeskeletal.py at', vqefile)

# 5. Core chemistry imports
from pyscf import gto, scf, ao2mo
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.quantum_info import SparsePauliOp

# 6. Import repo module
import vqeskeletal as vsk
importlib.reload(vsk)

# 7. Version report
print('\nVersions:')
for mod_name in ['qiskit','qiskit_nature','pyscf','qiskit_aer']:
    try:
        m = importlib.import_module(mod_name)
        print(f'  {mod_name:14s}:', getattr(m,'__version__','?'))
    except Exception as e:
        print(f'  {mod_name:14s}: MISSING ({e})')

# 8. Shared constants (NH3 active space 6 qubits)
NH3_GEOM = 'N 0 0 0; H 0.9377 0 -0.3816; H -0.4688 0.8119 -0.3816; H -0.4688 -0.8119 -0.3816'
ELECTRONS_ALPHA = 2
ELECTRONS_BETA  = 2
ACTIVE_SPATIAL_ORBS = 3  # -> 6 spin orbitals

print('\nEnvironment + repository initialization complete.')

Direct NH3 6-qubit active-space build and Pauli expansion (no error handling).

## Note
Padding adds zero-coefficient Pauli strings to reach the requested minimum; physics unaffected.

In [ ]:
# === 2. Build NH3 Active-Space Pauli Hamiltonian (Working Cell 2) ===
# Deterministic active-space construction (6 qubits) from PySCF integrals – NO FALLBACKS.

# SCF
mol = gto.M(atom=NH3_GEOM, basis='sto-3g', unit='Angstrom')
mf = scf.RHF(mol).run()
print(f'SCF energy: {mf.e_tot:.8f} Hartree')

# MO integrals
C = mf.mo_coeff
h_core_ao = mf.get_hcore()
h1_mo = C.T @ h_core_ao @ C
nmo = C.shape[1]
# 2-electron integrals (chemist) in MO basis
eri_mo_full = ao2mo.restore(1, ao2mo.full(mf._eri, C), nmo)

# Active space selection (first 3 spatial orbitals)
act = list(range(ACTIVE_SPATIAL_ORBS))
h1_act = h1_mo[np.ix_(act, act)]
eri_act = eri_mo_full[np.ix_(act, act, act, act)]

# Build ElectronicEnergy from raw integrals and wrap minimal problem info
# (We rely on qiskit-nature API directly; if this errors we STOP and fix.)
ee_act = ElectronicEnergy.from_raw_integrals(h1_act, eri_act)
problem_active = ElectronicStructureProblem(ee_act)

# Minimal wrapper supplying attributes AnsatzPlugin expects (accept flexible arg names)
class MiniProblem:
    def __init__(self, num_spin_orbitals=None, n_spin=None, num_alpha=None, n_alpha=None, num_beta=None, n_beta=None):
        self.num_spin_orbitals = num_spin_orbitals if num_spin_orbitals is not None else n_spin
        a = num_alpha if num_alpha is not None else n_alpha
        b = num_beta if num_beta is not None else n_beta
        self.num_particles = (a, b)
        if self.num_spin_orbitals is None or a is None or b is None:
            raise ValueError('MiniProblem requires spin orbitals and both particle counts.')

# Instantiate with explicit keywords
mini_problem = MiniProblem(num_spin_orbitals=2*ACTIVE_SPATIAL_ORBS,
                           num_alpha=ELECTRONS_ALPHA,
                           num_beta=ELECTRONS_BETA)

# Map fermionic Hamiltonian (robust to API return shape; still NO silent fallback)
mapper = JordanWignerMapper()
raw_ops = problem_active.second_q_ops()
print(f"second_q_ops() return type: {type(raw_ops)}")
if isinstance(raw_ops, dict):
    if 'ElectronicEnergy' not in raw_ops:
        raise KeyError("'ElectronicEnergy' key missing in second_q_ops() dict; keys: " + str(list(raw_ops.keys())))
    ferm_op = raw_ops['ElectronicEnergy']
elif isinstance(raw_ops, tuple):
    # Expect (main_op, aux_ops)
    if len(raw_ops) != 2:
        raise ValueError(f"Tuple from second_q_ops() length {len(raw_ops)} != 2; inspect manually.")
    ferm_op, aux_ops = raw_ops
    print(f"Extracted main fermionic operator from tuple; aux count: {len(aux_ops) if hasattr(aux_ops,'__len__') else 'N/A'}")
elif isinstance(raw_ops, list):
    if len(raw_ops) == 0:
        raise ValueError('Empty list from second_q_ops().')
    ferm_op = raw_ops[0]
    print('Warning: second_q_ops() returned list; using first element as main operator.')
else:
    raise TypeError(f"Unhandled type from second_q_ops(): {type(raw_ops)}")

qubit_op = mapper.map(ferm_op)

# Strict sanity checks (no silent fallbacks)
expected_qubits = 2 * ACTIVE_SPATIAL_ORBS
assert qubit_op.num_qubits == expected_qubits, f"Mapped qubits {qubit_op.num_qubits} != expected {expected_qubits}"
assert mini_problem.num_spin_orbitals == expected_qubits, "MiniProblem spin orbital mismatch"
assert sum(mini_problem.num_particles) == ELECTRONS_ALPHA + ELECTRONS_BETA, "Electron count mismatch"

# Term stats
labels = qubit_op.paulis.to_labels()
nonzero = [ (lbl, coeff) for lbl, coeff in zip(labels, qubit_op.coeffs) if abs(complex(coeff)) > 1e-12 ]
print(f"Qubits: {qubit_op.num_qubits}")
print(f"Non-zero Pauli terms: {len(nonzero)}")
print("Sample terms (first 10):")
for (lbl, coeff) in nonzero[:10]:
    print(f"  {coeff.real:+.8f} * {lbl}")

# System dictionary used downstream (NO fallbacks — this is the single source)
ham_system = {
    'problem_active': mini_problem,
    'mapper': mapper,
    'hamiltonian_active': qubit_op,
    'num_qubits': qubit_op.num_qubits,
    'basis': 'sto3g',
    'geometry': NH3_GEOM,
    'fallback': False
}
print('Hamiltonian system ready (use in later cells).')

### Cell 3 Description: UCCSD Ansatz Construction
Build the UCCSD excitation ansatz (with Hartree–Fock reference) sized to the previously created NH3 6‑qubit active-space Hamiltonian (`ham_system`).

Inputs: `ham_system` dictionary containing mapped Pauli Hamiltonian & particle/orbital counts.
Outputs: `uccsd_ansatz` (stored globally) ready for VQE; prints qubit count, parameter count, depth.
No fallbacks: raises if ansatz can't be constructed.


In [ ]:
# === 3. Build UCCSD Ansatz (Working Cell 3) ===
import importlib, vqeskeletal as vsk
importlib.reload(vsk)
from vqeskeletal import AnsatzPlugin

# Build ansatz from ham_system (expects mapper + mini problem)
ansatz = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=True)
ansatz.build_from_hamiltonian(ham_system)
info = ansatz.get_ansatz_info()
print('Ansatz info:', {k: info[k] for k in ['num_qubits','num_parameters','circuit_depth','vqe_ready']})

# Store for later VQE runs
uccsd_ansatz = ansatz
print('Stored as uccsd_ansatz.')

### Cell 3b Description: Optional Compact Ansatz Summary
Generates an auxiliary UCCSD ansatz using helper utilities (`groundstate` package functions) to display a concise circuit/parameter preview. Safe to skip; does not affect later VQE cells. Purely informational.


In [ ]:
# OPTIONAL: Compact UCCSD summary using provided helper utilities (repo already cloned)
import numpy as np, importlib
try:
    from groundstate import build_molecule_qubit_hamiltonian, uccsd_for_hamiltonian, circuit_summary
except ImportError:
    print('groundstate helpers not found (ensure repo cloned). Skipping summary.')
else:
    nh3_geom = NH3_GEOM
    ham = build_molecule_qubit_hamiltonian('NH3')
    ansatz_tmp, params_tmp = uccsd_for_hamiltonian(nh3_geom, ham, param_scale=0.02, seed=42)
    particles = ansatz_tmp.num_particles if isinstance(ansatz_tmp.num_particles,(tuple,list)) else (ansatz_tmp.num_particles, ansatz_tmp.num_particles)
    active_e = sum(particles)
    spatial = ansatz_tmp.num_spatial_orbitals
    print(f"Active space: {active_e} electrons, {spatial} orbitals -> {ansatz_tmp.num_qubits} qubits")
    print("Parameters:", np.array2string(params_tmp, separator=' ', max_line_width=120))
    print("\nCircuit (compact, high-level):")
    print(circuit_summary(ansatz_tmp, max_gates=25, decompose=False))

### VQE Skeleton Integration
Demonstrate integrating the repository's VQE skeleton (`vqeskeletal.py`) with a simple optimizer plugin.

This cell will:
1. Ensure the repo clone is present / updated.
2. Import the skeleton classes.
3. Define a minimal gradient-free optimizer (coordinate search) that fits the plugin interface.
4. Build the Hamiltonian + UCCSD ansatz via the skeleton plugins.
5. Run a mock VQE (note: expectation function is a placeholder returning 0.0 in the skeleton).

You can later replace the placeholder expectation with a real Estimator evaluation and plug in a hybrid (global→local) optimizer.

### Cell 4 Description: Coordinate-Descent VQE Integration
Rebuilds a fresh UCCSD ansatz plugin from `ham_system`, wraps the Hamiltonian with a direct plugin, performs an Estimator sanity check, then runs a lightweight coordinate-descent optimization (no stochastic SPSA here). Provides an initial exact estimator energy and improvement after local search. Raises if any estimator anomaly is detected.


In [ ]:
# === VQE Skeleton Integration (REAL Hamiltonian, NO FALLBACK) ===
# Uses the previously built `ham_system` (Cell 2) and `uccsd_ansatz` (Cell 3).
# If you see an energy near -5, the estimator path failed — we raise immediately.

import importlib, numpy as np, vqeskeletal as vsk
from vqeskeletal import AnsatzPlugin, ZNEDenoiserPlugin, VQE
from qiskit.quantum_info import SparsePauliOp

# Assert prerequisites
assert 'ham_system' in globals(), 'ham_system missing (run Cell 2)'
assert 'uccsd_ansatz' in globals(), 'uccsd_ansatz missing (run Cell 3)'

# Minimal direct Hamiltonian plugin wrapper (no internal reconstruction)
class DirectHamPlugin:
    def __init__(self, system_dict):
        self.system_dict = system_dict
    def get_hamiltonian(self):
        return self.system_dict

# Coordinate Descent Optimizer (unchanged logic, but clearer prints)
class CoordinateDescentOptimizer(vsk.ClassicalOptimizerPlugin):
    def __init__(self, max_iters=8, step=0.25, shrink=0.5, tol=1e-6, verbose=True):
        self.max_iters=max_iters; self.step=step; self.shrink=shrink; self.tol=tol; self.verbose=verbose
    def optimize(self, objective_function, initial_params):
        params = np.array(initial_params, dtype=float)
        best_val = objective_function(params); step=self.step
        if self.verbose: print(f'[CD] Initial energy: {best_val:.10f}')
        for it in range(self.max_iters):
            improved=False
            for i in range(len(params)):
                for d in (+1,-1):
                    trial = params.copy(); trial[i]+=d*step
                    val=objective_function(trial)
                    if val < best_val - self.tol:
                        best_val=val; params=trial; improved=True
                        if self.verbose: print(f'[CD] iter {it} param {i} {"+" if d>0 else "-"} -> {best_val:.10f}')
            if not improved:
                step*=self.shrink
                if self.verbose: print(f'[CD] No improvement; shrink step -> {step}')
                if step < self.tol:
                    if self.verbose: print('[CD] Converged (step below tol)')
                    break
        return params

# Rebuild ansatz plugin from real ham_system (independent copy so we don't mutate uccsd_ansatz)
ansatz_plugin = AnsatzPlugin(ansatz_reps=1, include_hf_state=True, verbose=False)
ansatz_plugin.build_from_hamiltonian(ham_system)
info = ansatz_plugin.get_ansatz_info()
print('Ansatz info (direct):', {k: info[k] for k in ['num_qubits','num_parameters','circuit_depth','vqe_ready']})
assert info['vqe_ready'], 'Ansatz not VQE-ready'

# Prepare plugins
ham_plugin = DirectHamPlugin(ham_system)
zne_plugin = ZNEDenoiserPlugin(noise_factors=[1.0], extrapolation_method='linear', verbose=False)
coord_opt = CoordinateDescentOptimizer(max_iters=8, step=0.25, shrink=0.5, tol=1e-6, verbose=True)

# Instantiate VQE (will re-build ansatz internally; acceptable) but we override with our built one
vqe_instance = VQE(ansatz_plugin, ham_plugin, coord_opt, zne_plugin, verbose=True)

# Initial parameters
init = ansatz_plugin.get_initial_parameters('zero')  # deterministic HF-based start

# Sanity: direct estimator energy BEFORE optimization
try:
    from qiskit_aer.primitives import Estimator
    est = Estimator()
    trial0 = ansatz_plugin.get_trial_wavefunction(init)
    e0 = est.run([trial0],[ham_system['hamiltonian_active']]).result().values[0]
    print(f'Initial estimator energy (raw) = {e0:.10f} Hartree')
    if abs(e0 + 5) < 0.5:
        raise RuntimeError('Estimator returned suspicious ~-5 energy (placeholder). Investigate measurement path.')
except Exception as ee:
    print('Estimator pre-check failed:', ee)
    raise

# Optimize
opt_params = coord_opt.optimize(vqe_instance.objective_function, init)
final_energy = vqe_instance.objective_function(opt_params)
print('\n[CoordinateDescent VQE] Final energy:', f'{final_energy:.10f}')
print('[CoordinateDescent VQE] Improvement:', f'{(e0-final_energy):.10f} Hartree')

# Expose for summary table
vqe_cd = vqe_instance
energy_cd = final_energy


### Cell 4a Description: Baseline Minimal VQE (No UCCSD)
Runs the simplest possible VQE loop on the NH3 Hamiltonian with a hardware-efficient layered Ry + CNOT ladder ansatz (depth=2 by default). Provides a quick reference energy and timing before UCCSD-based variants. Adjustable via NUM_LAYERS. No ZNE, no hybrid switching.

In [ ]:
# === 4a. Baseline Normal VQE (UCCSD HF Only, No Enhancements) ===
# Purpose: Provide a fair baseline using the SAME chemistry-aware UCCSD ansatz
# but with zero parameters (Hartree–Fock reference) and an optional tiny
# quick SPSA refinement (few iterations) WITHOUT hybrid switching or ZNE.
# This lets later variants (full SPSA, Hybrid, Hybrid+ZNE) show incremental gains.

from qiskit_aer.primitives import Estimator
from vqeskeletal import VQE, ZNEDenoiserPlugin, SPSAOptimizer
import numpy as np

assert 'uccsd_ansatz' in globals(), 'Run the UCCSD ansatz build cell first.'
assert 'ham_system' in globals(), 'Run the Hamiltonian build cell first.'

# 1. Hartree–Fock (zero-parameter) energy
hf_params = uccsd_ansatz.get_initial_parameters('zero')
est = Estimator()
hf_circuit = uccsd_ansatz.get_trial_wavefunction(hf_params)
hf_energy = est.run([hf_circuit],[ham_system['hamiltonian_active']]).result().values[0]
print(f"[Baseline HF] Energy (UCCSD ansatz, zero params / HF state): {hf_energy:.10f} Hartree")

# 2. Optional tiny plain SPSA (very few iterations) to show immediate improvement path
RUN_QUICK_OPT = True
quick_energy = hf_energy
quick_params = hf_params.copy()
if RUN_QUICK_OPT and len(hf_params) > 0:
    quick_spsa = SPSAOptimizer(max_iter=5, a=0.2, c=0.15, tol=1e-4, verbose=False)
    # Minimal wrapper VQE instance (ZNE disabled, no hybrid)
    class DirectHam: 
        def __init__(self, sysd): self.sysd = sysd
        def get_hamiltonian(self): return self.sysd
    no_zne_min = ZNEDenoiserPlugin(noise_factors=[1.0], verbose=False)
    vqe_min = VQE(uccsd_ansatz, DirectHam(ham_system), quick_spsa, no_zne_min, verbose=False)
    opt_params = quick_spsa.optimize(vqe_min.objective_function, hf_params)
    quick_energy = vqe_min.objective_function(opt_params)
    print(f"[Baseline Quick SPSA] Energy after {vqe_min.iteration_count} evals: {quick_energy:.10f} Hartree")
    print(f"[Baseline Quick SPSA] Improvement: {hf_energy - quick_energy:+.6f} Hartree")
    baseline_params = opt_params
else:
    baseline_params = hf_params

# Values captured for summary table
baseline_energy = float(quick_energy)


### Cell 5 Description: Basic VQE (SPSA Only)
Runs a standalone SPSA optimization on the UCCSD ansatz + NH3 Hamiltonian. Produces a noisy (stochastic) energy trajectory; serves as baseline for later hybrid refinement. ZNE disabled (single-factor identity).


In [ ]:
# === 5. Basic VQE (SPSA only, no ZNE) ===
from vqeskeletal import VQE, ZNEDenoiserPlugin, SPSAOptimizer

class DirectHam:
    def __init__(self, sysd): self.sysd = sysd
    def get_hamiltonian(self): return self.sysd

basic_optimizer = SPSAOptimizer(max_iter=25, a=0.25, c=0.15, tol=5e-4, verbose=True)
no_zne = ZNEDenoiserPlugin(noise_factors=[1.0], extrapolation_method='linear', verbose=False)

vqe_basic = VQE(uccsd_ansatz, DirectHam(ham_system), basic_optimizer, no_zne, verbose=True)
init_basic = uccsd_ansatz.get_initial_parameters('random_small')
params_basic, energy_basic = vqe_basic.run(init_basic)
print('\n[Basic VQE] Final energy:', energy_basic)

### Cell 6 Description: Hybrid VQE (SPSA → COBYLA)
Executes a two-phase optimization: global stochastic exploration via SPSA, then deterministic local refinement via COBYLA (forced switch). Tracks energy history to quantify improvement over pure SPSA.


In [ ]:
# === 6. Hybrid VQE (SPSA -> COBYLA, no ZNE) ===
from vqeskeletal import HybridSPSAThenCOBYLA, ZNEDenoiserPlugin, VQE

class DirectHam:
    def __init__(self, sysd): self.sysd = sysd
    def get_hamiltonian(self): return self.sysd

hybrid_opt = HybridSPSAThenCOBYLA(spsa_iters=40, switch_tol=5e-3, min_spsa=12, force_cobyla=True, verbose=True)
no_zne2 = ZNEDenoiserPlugin(noise_factors=[1.0], verbose=False)

vqe_hybrid = VQE(uccsd_ansatz, DirectHam(ham_system), hybrid_opt, no_zne2, verbose=True)
init_hybrid = uccsd_ansatz.get_initial_parameters('random_small')
params_hybrid, energy_hybrid = vqe_hybrid.run(init_hybrid)
print('\n[Hybrid VQE] Final energy:', energy_hybrid)

### Cell 7 Description: Hybrid VQE with ZNE (Richardson)
Repeats the hybrid optimization but wraps each objective evaluation with a Zero Noise Extrapolation plugin using scaling factors [1,3,5] and Richardson extrapolation. On an ideal simulator these factors yield identical energies; history helps when executing on noisy backends.


In [ ]:
# === 7. Hybrid VQE + ZNE (Richardson) ===
from vqeskeletal import ZNEDenoiserPlugin, VQE, HybridSPSAThenCOBYLA

class DirectHam:
    def __init__(self, sysd): self.sysd = sysd
    def get_hamiltonian(self): return self.sysd

zne_plugin = ZNEDenoiserPlugin(noise_factors=[1.0, 3.0, 5.0], extrapolation_method='richardson', verbose=True)
# Reuse hybrid optimizer settings
hybrid_opt2 = HybridSPSAThenCOBYLA(spsa_iters=40, switch_tol=5e-3, min_spsa=12, force_cobyla=True, verbose=True)

vqe_hybrid_zne = VQE(uccsd_ansatz, DirectHam(ham_system), hybrid_opt2, zne_plugin, verbose=True)
init_hybrid_zne = uccsd_ansatz.get_initial_parameters('random_small')
params_hybrid_zne, energy_hybrid_zne = vqe_hybrid_zne.run(init_hybrid_zne)
print('\n[Hybrid+ZNE VQE] Final (extrapolated) energy:', energy_hybrid_zne)
print('\nZNE analysis:', zne_plugin.get_zne_analysis())

### Cell 8 Description: Consolidated VQE Run Summary
Aggregates energies, best improvements, iteration counts, and parameter numbers for any executed VQE variants (Basic, Hybrid, Hybrid+ZNE). Safe to rerun; skips variants not yet executed. Also prints ZNE extrapolation details if available.


In [ ]:
# === 8. VQE Results Summary (Energies & Iterations) ===
# Collect and print final energies + iteration counts for each executed variant.
# Safe to run multiple times after any subset of runs.

summary_rows = []
from math import isnan

# Helper: build lightweight object for baseline (non-VQE) run so we can reuse collect()
if 'baseline_energy' in globals() and 'baseline_params' in globals():
    class _BaselineObj:
        def __init__(self, energy, params):
            self.energy_history = [float(energy)]  # single evaluation (or quick mini optimization)
            self.iteration_count = len(self.energy_history)
            class _AnsatzWrap:
                def __init__(self, n): self.num_parameters = n
            self.ansatz_plugin = _AnsatzWrap(len(params))
    baseline_obj = _BaselineObj(baseline_energy, baseline_params)
else:
    baseline_obj = None

def collect(label, vqe_obj, energy_name, energy_val):
    if vqe_obj is None:
        return
    # Final recorded energy; fall back to provided energy_val
    final_energy = float(energy_val) if energy_val is not None else (
        vqe_obj.energy_history[-1] if vqe_obj.energy_history else float('nan')
    )
    best_energy = min(vqe_obj.energy_history) if vqe_obj.energy_history else final_energy
    iters = getattr(vqe_obj, 'iteration_count', len(getattr(vqe_obj,'energy_history', [])))
    params = getattr(getattr(vqe_obj, 'ansatz_plugin', object()), 'num_parameters', None)
    summary_rows.append({
        'variant': label,
        'final_energy': final_energy,
        'best_energy': best_energy,
        'iterations': iters,
        'parameters': params,
        'improvement': (vqe_obj.energy_history[0] - best_energy) if len(vqe_obj.energy_history) >= 2 else 0.0
    })

# Baselines / variants
collect('Baseline HF', baseline_obj, 'baseline_energy', globals().get('baseline_energy'))
collect('CoordinateDescent', globals().get('vqe_cd'), 'energy_cd', globals().get('energy_cd'))
collect('Basic (SPSA)', globals().get('vqe_basic'), 'energy_basic', globals().get('energy_basic'))
collect('Hybrid (SPSA->COBYLA)', globals().get('vqe_hybrid'), 'energy_hybrid', globals().get('energy_hybrid'))
collect('Hybrid+ZNE', globals().get('vqe_hybrid_zne'), 'energy_hybrid_zne', globals().get('energy_hybrid_zne'))

if not summary_rows:
    print('No VQE runs detected yet. Run the variant cells first.')
else:
    # Pretty print
    print('\n=== VQE Run Summary ===')
    header = f"{'Variant':28s} {'Final Energy (Ha)':>18s} {'Best (Ha)':>14s} {'Δ (Ha)':>12s} {'Iters':>7s} {'Params':>7s}"""
    print(header)
    print('-'*len(header))
    for row in summary_rows:
        dE = row['improvement']
        print(f"{row['variant']:28s} {row['final_energy']:18.10f} {row['best_energy']:14.10f} {dE:12.6f} {row['iterations']:7d} {row['parameters']:7}")
    print('\nNotes:')
    print('  Δ (Ha) = initial_energy - best_energy (if multiple evaluations).')
    print('  Best vs Final can differ if last evaluation was not the minimum encountered.')
    if 'zne_plugin' in globals():
        zne_analysis = zne_plugin.get_zne_analysis()
        if 'error' not in zne_analysis:
            print('\nZNE summary (last run):')
            for k,v in zne_analysis.items():
                if isinstance(v, float):
                    print(f"  {k}: {v}")
                else:
                    print(f"  {k}: {v}")
        else:
            print('\nZNE summary: No multi-noise measurements collected (single-factor only).')
